In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)

LIB_DIR = PROJECT_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import research_statistics as rstats

TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

print("统计工具库：", rstats.__file__)

In [ ]:
# 09.1.1：综合盈利因子的 HAC 与逐年样本外稳定性检验
IC_PATH = (
    TABLE_DIR
    / "05_integrated_profitability_ic_rankic_timeseries.csv"
)

QUINTILE_PATH = (
    TABLE_DIR
    / "05_integrated_profitability_quintile_return_timeseries.csv"
)

ic_rankic = pd.read_csv(IC_PATH)
quintile_returns = pd.read_csv(QUINTILE_PATH)

ic_rankic["signal_date"] = pd.to_datetime(ic_rankic["signal_date"])
quintile_returns["signal_date"] = pd.to_datetime(
    quintile_returns["signal_date"]
)

monthly_result = (
    ic_rankic[
        ["signal_date", "ic", "rank_ic"]
    ]
    .merge(
        quintile_returns[
            ["signal_date", "high_minus_low_return"]
        ],
        on="signal_date",
        how="inner",
    )
    .dropna()
    .sort_values("signal_date")
)

HAC_MAX_LAGS = 1

period_masks = {
    "all_available_2019_2025": (
        monthly_result["signal_date"]
        <= pd.Timestamp("2025-12-31")
    ),
    "in_sample_available_2019_2022": (
        monthly_result["signal_date"]
        <= pd.Timestamp("2022-12-31")
    ),
    "out_of_sample_2023_2025": (
        (monthly_result["signal_date"] >= pd.Timestamp("2023-01-01"))
        & (monthly_result["signal_date"] <= pd.Timestamp("2025-12-31"))
    ),
    "out_of_sample_2023": (
        monthly_result["signal_date"].dt.year == 2023
    ),
    "out_of_sample_2024": (
        monthly_result["signal_date"].dt.year == 2024
    ),
    "out_of_sample_2025": (
        monthly_result["signal_date"].dt.year == 2025
    ),
}

stability_rows = []

for period_name, period_mask in period_masks.items():
    period_data = monthly_result.loc[period_mask].copy()

    row = {
        "period": period_name,
        "observations": len(period_data),
        "positive_rank_ic_ratio": (
            period_data["rank_ic"] > 0
        ).mean(),
    }

    for metric_name in ["ic", "rank_ic", "high_minus_low_return"]:
        hac_result = rstats.hac_mean_test(
            period_data[metric_name],
            max_lags=HAC_MAX_LAGS,
        )

        row[metric_name + "_mean"] = hac_result["mean"]
        row[metric_name + "_hac_t_stat"] = hac_result["hac_t_stat"]
        row[metric_name + "_hac_p_value"] = hac_result["hac_p_value"]

    stability_rows.append(row)

hac_stability_summary = pd.DataFrame(stability_rows)

SUMMARY_PATH = (
    TABLE_DIR
    / "09_integrated_profitability_hac_stability_summary.csv"
)

RULES_PATH = (
    METADATA_DIR
    / "09_integrated_profitability_hac_stability_rules.json"
)

FIGURE_PATH = (
    FIGURE_DIR
    / "09_integrated_profitability_oos_yearly_stability.png"
)

hac_stability_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

rules = {
    "factor": "integrated_profitability_structure_proxy_strict",
    "monthly_return_horizon": "T_plus_1_open_to_T_plus_21_open",
    "hac_max_lags": HAC_MAX_LAGS,
    "reason_for_hac": (
        "月度信号的 20 个交易日收益窗口存在相邻月份重叠，"
        "因此对均值检验使用 HAC 标准误。"
    ),
    "out_of_sample_years": [2023, 2024, 2025],
    "no_parameter_optimization": True,
}

with RULES_PATH.open("w", encoding="utf-8") as file:
    json.dump(rules, file, ensure_ascii=False, indent=2)

oos_yearly = hac_stability_summary[
    hac_stability_summary["period"].isin(
        [
            "out_of_sample_2023",
            "out_of_sample_2024",
            "out_of_sample_2025",
        ]
    )
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    oos_yearly["period"],
    oos_yearly["rank_ic_mean"],
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Out-of-Sample Mean RankIC by Year")
axes[0].set_ylabel("Mean RankIC")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    oos_yearly["period"],
    oos_yearly["high_minus_low_return_mean"],
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Out-of-Sample Mean Q5 - Q1 Return by Year")
axes[1].set_ylabel("Mean 20-Day Return")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("===== HAC 与逐年稳定性检验完成 =====")
print("汇总表：", SUMMARY_PATH)
print("规则说明：", RULES_PATH)
print("图表：", FIGURE_PATH)

display(hac_stability_summary)

In [ ]:
# 09.2.0：只读检查 RQAlpha 正式回测产物字段 ----
# 请确认已经运行本 Notebook 最开始的 09.0 初始化单元。

NAV_PATH = (
    PROJECT_ROOT / "artifacts" / "tables"
    / "07_rqalpha_oos_q5_daily_nav.csv"
)
DAILY_EXECUTION_PATH = (
    PROJECT_ROOT / "artifacts" / "tables"
    / "07_rqalpha_oos_q5_daily_execution.csv"
)
EXECUTION_SUMMARY_PATH = (
    PROJECT_ROOT / "artifacts" / "tables"
    / "07_rqalpha_oos_q5_execution_summary.csv"
)

for path in [NAV_PATH, DAILY_EXECUTION_PATH, EXECUTION_SUMMARY_PATH]:
    assert path.exists(), f"未找到正式回测产物：{path}"

nav_preview = pd.read_csv(NAV_PATH)
execution_preview = pd.read_csv(DAILY_EXECUTION_PATH)
execution_summary_preview = pd.read_csv(EXECUTION_SUMMARY_PATH)

print("===== 逐日净值字段 =====")
print(nav_preview.columns.tolist())
display(nav_preview.head(3))

print("\n===== 逐日执行字段 =====")
print(execution_preview.columns.tolist())
display(execution_preview.head(3))

print("\n===== 执行汇总字段 =====")
print(execution_summary_preview.columns.tolist())
display(execution_summary_preview)

In [ ]:
%%writefile /home/rice/notebook/a-share-profitability-factor-replication/lib/research_statistics.py
import numpy as np
import pandas as pd
import statsmodels.api as sm


def hac_mean_test(values, max_lags=1):
    """对一组时间序列均值进行 HAC/Newey-West 检验。"""
    series = pd.Series(values).dropna().astype(float)

    if len(series) < 2:
        raise ValueError("HAC 检验至少需要两个有效观测。")

    lags = min(int(max_lags), len(series) - 1)
    model = sm.OLS(series.to_numpy(), np.ones((len(series), 1))).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": lags},
    )

    return {
        "observations": int(len(series)),
        "mean": float(series.mean()),
        "hac_t_stat": float(model.tvalues[0]),
        "hac_p_value": float(model.pvalues[0]),
        "max_lags": int(lags),
    }


def nav_drawdown_diagnostics(nav):
    """计算净值序列的回撤路径及最大回撤事件。"""
    series = pd.Series(nav).dropna().astype(float).copy()
    series.index = pd.to_datetime(series.index)
    series = series[~series.index.duplicated(keep="last")].sort_index()

    if series.empty or (series <= 0).any():
        raise ValueError("净值序列必须非空且全部大于 0。")

    running_peak = series.cummax()
    drawdown = series.div(running_peak).sub(1.0)

    trough_date = drawdown.idxmin()
    peak_date = series.loc[:trough_date].idxmax()
    peak_nav = float(series.loc[peak_date])
    trough_nav = float(series.loc[trough_date])

    post_trough = series.loc[trough_date:]
    recovery_candidates = post_trough[post_trough >= peak_nav]
    recovery_date = (
        recovery_candidates.index[0]
        if len(recovery_candidates) > 0
        else pd.NaT
    )

    summary = {
        "peak_date": peak_date,
        "trough_date": trough_date,
        "recovery_date": recovery_date,
        "peak_nav": peak_nav,
        "trough_nav": trough_nav,
        "max_drawdown": float(drawdown.loc[trough_date]),
        "peak_to_trough_observations": int(
            series.index.get_loc(trough_date) - series.index.get_loc(peak_date)
        ),
        "trough_recovered": bool(pd.notna(recovery_date)),
        "trough_to_recovery_observations": (
            int(series.index.get_loc(recovery_date) - series.index.get_loc(trough_date))
            if pd.notna(recovery_date)
            else None
        ),
    }

    return drawdown, summary


def period_end_returns(nav, frequency="M"):
    """将逐日净值转换为指定频率的区间收益。"""
    series = pd.Series(nav).dropna().astype(float).copy()
    series.index = pd.to_datetime(series.index)
    series = series[~series.index.duplicated(keep="last")].sort_index()

    period_end_nav = series.resample(frequency).last()
    return period_end_nav.pct_change()

In [ ]:
import importlib

In [ ]:
# 09.2：RQAlpha 样本外失败案例、回撤与执行归因 ----
import research_statistics as rstats
rstats = importlib.reload(rstats)

TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

NAV_PATH = TABLE_DIR / "07_rqalpha_oos_q5_daily_nav.csv"
DAILY_EXECUTION_PATH = TABLE_DIR / "07_rqalpha_oos_q5_daily_execution.csv"
EXECUTION_SUMMARY_PATH = TABLE_DIR / "07_rqalpha_oos_q5_execution_summary.csv"

INITIAL_CASH = 100_000_000.0

nav = pd.read_csv(NAV_PATH, parse_dates=["date"]).set_index("date").sort_index()
execution = pd.read_csv(
    DAILY_EXECUTION_PATH,
    parse_dates=["trade_date"],
).sort_values("trade_date")
execution_summary = pd.read_csv(EXECUTION_SUMMARY_PATH).iloc[0]

required_nav_columns = {
    "strategy_nav",
    "benchmark_nav",
    "relative_nav",
    "strategy_drawdown",
    "benchmark_drawdown",
}
assert required_nav_columns.issubset(nav.columns), "逐日净值字段不完整。"

# 1. 最大回撤事件：策略、基准、策略相对基准
drawdown_records = []
derived_drawdowns = {}

for series_name, nav_column in [
    ("Q5_strategy", "strategy_nav"),
    ("CSI_800_benchmark", "benchmark_nav"),
    ("Q5_relative_to_CSI_800", "relative_nav"),
]:
    drawdown, summary = rstats.nav_drawdown_diagnostics(nav[nav_column])
    derived_drawdowns[series_name] = drawdown

    summary["series"] = series_name
    drawdown_records.append(summary)

drawdown_episode = pd.DataFrame(drawdown_records)

for column in ["peak_date", "trough_date", "recovery_date"]:
    drawdown_episode[column] = pd.to_datetime(
        drawdown_episode[column]
    ).dt.strftime("%Y-%m-%d")

drawdown_episode["max_drawdown_pct"] = (
    drawdown_episode["max_drawdown"] * 100
)

strategy_drawdown_audit = float(
    (
        nav["strategy_drawdown"]
        - derived_drawdowns["Q5_strategy"]
    ).abs().max()
)

# 2. 月度表现：策略、基准、相对净值
monthly_nav = nav[
    ["strategy_nav", "benchmark_nav", "relative_nav"]
].resample("M").last()

monthly_performance = monthly_nav.pct_change()
monthly_performance.iloc[0] = monthly_nav.iloc[0] - 1.0
monthly_performance = monthly_performance.rename(
    columns={
        "strategy_nav": "strategy_return",
        "benchmark_nav": "benchmark_return",
        "relative_nav": "relative_return",
    }
)
monthly_performance["calendar_month"] = monthly_performance.index.strftime("%Y-%m")

# 3. 按自然年度汇总；2026 仅为最后一期持有窗口尾部
yearly_records = []
previous_nav = {
    "strategy_nav": 1.0,
    "benchmark_nav": 1.0,
    "relative_nav": 1.0,
}

for calendar_year, frame in nav.groupby(nav.index.year):
    final_nav = frame[
        ["strategy_nav", "benchmark_nav", "relative_nav"]
    ].iloc[-1]

    yearly_records.append(
        {
            "period": (
                str(calendar_year)
                if calendar_year < 2026
                else "2026_exit_tail"
            ),
            "trading_day_count": int(len(frame)),
            "strategy_return": (
                final_nav["strategy_nav"]
                / previous_nav["strategy_nav"]
                - 1.0
            ),
            "benchmark_return": (
                final_nav["benchmark_nav"]
                / previous_nav["benchmark_nav"]
                - 1.0
            ),
            "relative_return": (
                final_nav["relative_nav"]
                / previous_nav["relative_nav"]
                - 1.0
            ),
        }
    )

    previous_nav = final_nav.to_dict()

yearly_performance = pd.DataFrame(yearly_records)

# 4. 成交、成本与换手摘要
execution["transaction_cost"] = (
    execution["commission"] + execution["tax"]
)
execution["transaction_cost_bps_of_notional"] = (
    execution["transaction_cost"]
    .div(execution["filled_notional"].where(execution["filled_notional"] > 0))
    .mul(10_000)
)

monthly_execution = (
    execution.set_index("trade_date")
    .resample("M")[[
        "trade_record_count",
        "filled_notional",
        "commission",
        "tax",
        "transaction_cost",
    ]]
    .sum()
    .reset_index()
)

monthly_execution["transaction_cost_bps_of_notional"] = (
    monthly_execution["transaction_cost"]
    .div(
        monthly_execution["filled_notional"].where(
            monthly_execution["filled_notional"] > 0
        )
    )
    .mul(10_000)
)

execution_diagnostics = pd.DataFrame(
    [{
        "backtest_start_date": nav.index.min().strftime("%Y-%m-%d"),
        "backtest_end_date": nav.index.max().strftime("%Y-%m-%d"),
        "trading_day_count": int(len(nav)),
        "trade_record_count": int(execution_summary["trade_record_count"]),
        "trade_day_count": int(execution_summary["trade_day_count"]),
        "total_filled_notional": float(
            execution_summary["total_filled_notional"]
        ),
        "total_notional_to_initial_cash_multiple": float(
            execution_summary["total_filled_notional"] / INITIAL_CASH
        ),
        "total_transaction_cost": float(
            execution_summary["total_transaction_cost"]
        ),
        "transaction_cost_pct_of_initial_cash": float(
            execution_summary["total_transaction_cost"] / INITIAL_CASH
        ),
        "transaction_cost_bps_of_filled_notional": float(
            execution_summary["total_transaction_cost"]
            / execution_summary["total_filled_notional"]
            * 10_000
        ),
        "largest_single_trade_day_notional": float(
            execution["filled_notional"].max()
        ),
        "largest_single_trade_day_cost": float(
            execution["transaction_cost"].max()
        ),
        "strategy_drawdown_recalculation_max_abs_difference": (
            strategy_drawdown_audit
        ),
    }]
)

# 5. 安全产物
drawdown_episode.to_csv(
    TABLE_DIR / "09_rqalpha_oos_q5_max_drawdown_episode.csv",
    index=False,
)
monthly_performance.reset_index(names="month_end").to_csv(
    TABLE_DIR / "09_rqalpha_oos_q5_monthly_performance.csv",
    index=False,
)
yearly_performance.to_csv(
    TABLE_DIR / "09_rqalpha_oos_q5_calendar_year_performance.csv",
    index=False,
)
monthly_execution.to_csv(
    TABLE_DIR / "09_rqalpha_oos_q5_monthly_execution.csv",
    index=False,
)
execution_diagnostics.to_csv(
    TABLE_DIR / "09_rqalpha_oos_q5_execution_diagnostics.csv",
    index=False,
)

rules = {
    "analysis_type": "read_only_post_backtest_diagnostics",
    "backtest_engine": "RQAlpha",
    "source_nav_file": str(NAV_PATH),
    "source_execution_file": str(DAILY_EXECUTION_PATH),
    "initial_cash": INITIAL_CASH,
    "drawdown_method": "running_peak_drawdown",
    "relative_nav_definition": "strategy_nav / benchmark_nav",
    "important_limitations": [
        "该分析不重新调参、不修改因子、也不重跑回测。",
        "期末存在涨跌停拒单时，最终净值为持仓市值计价，不等同于完全清仓后的现金收益。",
        "零股下单提示与涨跌停拒单数量未形成独立的逐笔拒单审计表，不能由成交表反推出精确次数。",
    ],
}
with open(
    METADATA_DIR / "09_rqalpha_oos_q5_failure_analysis_rules.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(rules, file, ensure_ascii=False, indent=2)

# 6. 图表
plt.style.use("default")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(
    nav.index,
    nav["strategy_nav"] / nav["strategy_nav"].iloc[0],
    label="Q5 strategy",
)
axes[0, 0].plot(
    nav.index,
    nav["benchmark_nav"] / nav["benchmark_nav"].iloc[0],
    label="CSI 800",
)
axes[0, 0].set_title("RQAlpha OOS NAV")
axes[0, 0].set_ylabel("Normalized NAV")
axes[0, 0].legend()

axes[0, 1].fill_between(
    nav.index,
    nav["strategy_drawdown"],
    0,
    alpha=0.35,
    label="Q5 strategy",
)
axes[0, 1].plot(
    nav.index,
    nav["benchmark_drawdown"],
    label="CSI 800",
)
axes[0, 1].set_title("Drawdown Comparison")
axes[0, 1].set_ylabel("Drawdown")
axes[0, 1].legend()

axes[1, 0].bar(
    monthly_performance.index,
    monthly_performance["relative_return"] * 100,
    width=20,
)
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set_title("Monthly Relative Return: Q5 / CSI 800")
axes[1, 0].set_ylabel("Return (%)")

axes[1, 1].plot(
    monthly_execution["trade_date"],
    monthly_execution["transaction_cost"].cumsum() / 1_000_000,
)
axes[1, 1].set_title("Cumulative Transaction Cost")
axes[1, 1].set_ylabel("Cost (million CNY)")

fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "09_rqalpha_oos_q5_failure_analysis.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("===== 最大回撤事件 =====")
display(drawdown_episode)

print("\n===== 自然年度表现 =====")
display(yearly_performance)

print("\n===== 执行与成本诊断 =====")
display(execution_diagnostics)

print("\n===== 09.2 已保存 =====")
print(TABLE_DIR / "09_rqalpha_oos_q5_max_drawdown_episode.csv")
print(TABLE_DIR / "09_rqalpha_oos_q5_monthly_performance.csv")
print(TABLE_DIR / "09_rqalpha_oos_q5_calendar_year_performance.csv")
print(TABLE_DIR / "09_rqalpha_oos_q5_monthly_execution.csv")
print(TABLE_DIR / "09_rqalpha_oos_q5_execution_diagnostics.csv")
print(FIGURE_DIR / "09_rqalpha_oos_q5_failure_analysis.png")

In [ ]:
import pickle
import rqdatac

In [ ]:
# 09.3.0：期末强制清仓敏感性检验的只读预检查 ----
INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"

TARGET_FILES = {
    "Q5_factor_high": (
        INPUT_DIR / "07_oos_q5_target_weights_portable.pkl"
    ),
    "Q1_factor_low": (
        INPUT_DIR / "07_q1_long_control_target_weights_portable.pkl"
    ),
    "Equal_weight_eligible_universe": (
        INPUT_DIR
        / "07_equal_weight_eligible_universe_target_weights_portable.pkl"
    ),
}

preflight_records = []
final_target_dates = []

for portfolio_name, target_file in TARGET_FILES.items():
    assert target_file.exists(), f"未找到冻结权重文件：{target_file}"

    with open(target_file, "rb") as file:
        payload = pickle.load(file)

    assert isinstance(payload, dict), f"{portfolio_name} 的权重文件不是字典。"
    assert {"dates", "assets", "weights"}.issubset(payload), (
        f"{portfolio_name} 的权重文件字段不完整。"
    )

    target_dates = pd.to_datetime(payload["dates"]).normalize()
    final_target_date = target_dates.max()
    final_target_dates.append(final_target_date)

    preflight_records.append(
        {
            "portfolio": portfolio_name,
            "event_date_count": int(len(target_dates)),
            "asset_column_count": int(len(payload["assets"])),
            "first_target_date": target_dates.min().strftime("%Y-%m-%d"),
            "final_target_date": final_target_date.strftime("%Y-%m-%d"),
        }
    )

assert len(set(final_target_dates)) == 1, (
    "三个组合的最终目标日期不一致，不能进行公平的强制清仓比较。"
)

last_target_date = final_target_dates[0]
future_calendar = pd.to_datetime(
    rqdatac.get_trading_dates(
        last_target_date + pd.Timedelta(days=1),
        last_target_date + pd.Timedelta(days=90),
    )
).normalize()

forced_exit_dates = future_calendar[:20]
assert len(forced_exit_dates) == 20, "后续可用交易日不足 20 个。"

preflight = pd.DataFrame(preflight_records)

print("===== 09.3 强制清仓预检查 =====")
display(preflight)

print("最后一个原始目标日：", last_target_date.strftime("%Y-%m-%d"))
print("拟追加的首次强制退出日：", forced_exit_dates[0].strftime("%Y-%m-%d"))
print("拟追加的最后强制退出日：", forced_exit_dates[-1].strftime("%Y-%m-%d"))
print("追加零权重交易日数：", len(forced_exit_dates))

In [ ]:
# 09.3.1：生成三个组合的 20 日强制清仓权重输入 ----
INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"

SOURCE_AND_OUTPUT = {
    "Q5_factor_high": {
        "source": INPUT_DIR / "07_oos_q5_target_weights_portable.pkl",
        "output": (
            INPUT_DIR
            / "09_q5_factor_high_forced_exit_20d_target_weights_portable.pkl"
        ),
    },
    "Q1_factor_low": {
        "source": INPUT_DIR / "07_q1_long_control_target_weights_portable.pkl",
        "output": (
            INPUT_DIR
            / "09_q1_factor_low_forced_exit_20d_target_weights_portable.pkl"
        ),
    },
    "Equal_weight_eligible_universe": {
        "source": (
            INPUT_DIR
            / "07_equal_weight_eligible_universe_target_weights_portable.pkl"
        ),
        "output": (
            INPUT_DIR
            / "09_equal_weight_eligible_universe_forced_exit_20d_target_weights_portable.pkl"
        ),
    },
}

original_end_dates = []
prepared_records = []

for portfolio_name, paths in SOURCE_AND_OUTPUT.items():
    with open(paths["source"], "rb") as file:
        source_payload = pickle.load(file)

    source_dates = pd.to_datetime(source_payload["dates"]).normalize()
    assets = list(source_payload["assets"])
    source_weights = list(source_payload["weights"])

    assert len(source_dates) == len(source_weights), (
        f"{portfolio_name} 的日期数与权重行数不一致。"
    )
    assert source_dates.is_unique, f"{portfolio_name} 存在重复目标日期。"
    assert all(len(row) == len(assets) for row in source_weights), (
        f"{portfolio_name} 的权重列数与股票数不一致。"
    )

    original_end_dates.append(source_dates.max())

assert len(set(original_end_dates)) == 1, "三个原始权重终点不一致。"

original_end_date = original_end_dates[0]
future_dates = pd.to_datetime(
    rqdatac.get_trading_dates(
        original_end_date + pd.Timedelta(days=1),
        original_end_date + pd.Timedelta(days=90),
    )
).normalize()[:20]

assert len(future_dates) == 20, "无法取得完整的 20 个后续交易日。"

for portfolio_name, paths in SOURCE_AND_OUTPUT.items():
    with open(paths["source"], "rb") as file:
        source_payload = pickle.load(file)

    source_dates = pd.to_datetime(source_payload["dates"]).normalize()
    assets = list(source_payload["assets"])
    source_weights = list(source_payload["weights"])

    zero_weight_row = [0.0] * len(assets)
    extended_dates = list(source_dates) + list(future_dates)
    extended_weights = source_weights + [zero_weight_row] * len(future_dates)

    portable_payload = {
        "format": "rqalpha_target_weights_portable_v1",
        "dates": [date.strftime("%Y-%m-%d") for date in extended_dates],
        "assets": assets,
        "weights": extended_weights,
    }

    with open(paths["output"], "wb") as file:
        pickle.dump(portable_payload, file, protocol=4)

    prepared_records.append(
        {
            "portfolio": portfolio_name,
            "original_event_date_count": len(source_dates),
            "forced_exit_zero_weight_dates": len(future_dates),
            "extended_event_date_count": len(extended_dates),
            "original_final_date": original_end_date.strftime("%Y-%m-%d"),
            "forced_exit_start_date": future_dates[0].strftime("%Y-%m-%d"),
            "forced_exit_end_date": future_dates[-1].strftime("%Y-%m-%d"),
            "asset_count": len(assets),
            "portable_input_file": str(paths["output"]),
        }
    )

prepared_summary = pd.DataFrame(prepared_records)

print("===== 09.3 强制清仓输入已生成 =====")
display(prepared_summary)

In [ ]:
# 09.3.2：RQAlpha 三组合强制清仓尾部敏感性回测（完整可重跑版） ----
from pathlib import Path
import sys
import json
import pickle
import importlib

import pandas as pd

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)
LIB_DIR = PROJECT_ROOT / "lib"
INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"
BACKTEST_DIR = PROJECT_ROOT / "data" / "backtests"
NATIVE_CACHE_DIR = PROJECT_ROOT / "cache" / "rqalpha_native"

BACKTEST_DIR.mkdir(parents=True, exist_ok=True)
NATIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

BACKTEST_SPECS = [
    {
        "portfolio": "Q5_factor_high",
        "target_file": (
            INPUT_DIR
            / "09_q5_factor_high_forced_exit_20d_target_weights_portable.pkl"
        ),
        "result_file": (
            BACKTEST_DIR
            / "09_rqalpha_q5_forced_exit_20d_portable_result.pkl"
        ),
    },
    {
        "portfolio": "Q1_factor_low",
        "target_file": (
            INPUT_DIR
            / "09_q1_factor_low_forced_exit_20d_target_weights_portable.pkl"
        ),
        "result_file": (
            BACKTEST_DIR
            / "09_rqalpha_q1_forced_exit_20d_portable_result.pkl"
        ),
    },
    {
        "portfolio": "Equal_weight_eligible_universe",
        "target_file": (
            INPUT_DIR
            / "09_equal_weight_eligible_universe_forced_exit_20d_target_weights_portable.pkl"
        ),
        "result_file": (
            BACKTEST_DIR
            / "09_rqalpha_equal_weight_forced_exit_20d_portable_result.pkl"
        ),
    },
]

# Q5 刚刚已经完成，只是旧版序列化 trades 时出错。
# 先保留它的内存结果，避免重新跑完整三年。
completed_q5_result = None
q5_result_file = BACKTEST_SPECS[0]["result_file"]

if not q5_result_file.exists():
    assert "result" in globals(), (
        "未找到内存中的 Q5 RQAlpha 结果。"
        "如果内核已重启，只能重新运行 Q5。"
    )
    assert isinstance(result, dict) and "sys_analyser" in result, (
        "当前内存 result 不是有效的 Q5 RQAlpha 结果。"
    )
    completed_q5_result = result
    print("已识别内存中完成的 Q5 回测；将直接保存，不重复回测。")

for sequence, spec in enumerate(BACKTEST_SPECS, start=1):
    print("\n" + "=" * 60)
    print("强制清仓回测 {}/3：{}".format(sequence, spec["portfolio"]))

    if spec["result_file"].exists():
        print("本地可移植结果缓存命中，跳过回测。")
        continue

    if spec["portfolio"] == "Q5_factor_high":
        backtest_result = completed_q5_result
        print("阶段 1/3：复用刚刚完成的 Q5 内存结果。")
    else:
        assert spec["target_file"].exists(), (
            "未找到权重输入：{}".format(spec["target_file"])
        )

        with open(spec["target_file"], "rb") as file:
            target_payload = pickle.load(file)

        target_weights = pd.DataFrame(
            target_payload["weights"],
            index=pd.to_datetime(target_payload["dates"]),
            columns=target_payload["assets"],
            dtype=float,
        ).sort_index()

        assert target_weights.index.is_unique, "目标权重日期存在重复。"
        assert (target_weights.iloc[-20:].abs().sum(axis=1) == 0).all(), (
            "最后 20 个交易日必须全部为零权重。"
        )

        print(
            "阶段 1/3：已读取权重；日期 {} 至 {}。".format(
                target_weights.index.min().date(),
                target_weights.index.max().date(),
            )
        )

        # 每次运行前新导入；禁止使用 importlib.reload。
        sys.modules.pop("rqalpha_strategy_utils", None)
        sys.modules.pop("rqalpha.user_module", None)
        importlib.invalidate_caches()
        rau = importlib.import_module("rqalpha_strategy_utils")

        print("阶段 2/3：运行 RQAlpha，日志每 20 个交易日更新一次。")

        backtest_result = rau.run_target_weight_backtest(
            target_weights=target_weights,
            initial_cash=100_000_000,
            benchmark="000906.XSHG",
            analyser_output_file=str(
                NATIVE_CACHE_DIR
                / "09_{}_forced_exit_native.pkl".format(spec["portfolio"])
            ),
            plot_output_path=str(
                NATIVE_CACHE_DIR
                / "09_{}_forced_exit_plots".format(spec["portfolio"])
            ),
            matching_type="current_bar",
            min_commission=5.0,
            stock_commission_multiplier=0.375,
            tax_multiplier=1.0,
            price_limit=True,
            volume_limit=True,
            volume_percent=0.05,
            show_progress=True,
        )

    assert isinstance(backtest_result, dict), (
        "{} 未返回字典结果。".format(spec["portfolio"])
    )
    assert "sys_analyser" in backtest_result, (
        "{} 未返回 sys_analyser。".format(spec["portfolio"])
    )

    analyser = backtest_result["sys_analyser"]

    # 处理 datetime 同名索引与字段，避免 reset_index 冲突。
    portable_tables = {}

    for table_name in [
        "portfolio",
        "benchmark_portfolio",
        "stock_account",
        "trades",
        "stock_positions",
    ]:
        table = analyser[table_name].copy()

        if table.index.name is not None and table.index.name in table.columns:
            table = table.reset_index(drop=True)
        else:
            table = table.reset_index()

        portable_tables[table_name] = json.loads(
            table.to_json(orient="split", date_format="iso")
        )

    portable_result = {
        "format": "rqalpha_analyser_portable_v1",
        "portfolio_name": spec["portfolio"],
        "summary": json.loads(
            json.dumps(analyser["summary"], default=str)
        ),
    }
    portable_result.update(portable_tables)

    with open(spec["result_file"], "wb") as file:
        pickle.dump(portable_result, file, protocol=4)

    print("阶段 3/3：已保存可移植结果：", spec["result_file"])

print("\n===== 09.3 三组合强制清仓回测完成 =====")

In [ ]:
# 09.3.2a：保存已完成但尚未序列化的 Q5 强制清仓结果 ----
Q5_PORTABLE_RESULT = (
    PROJECT_ROOT
    / "data" / "backtests"
    / "09_rqalpha_q5_forced_exit_20d_portable_result.pkl"
)

assert "result" in globals(), "当前内核中找不到 result；请勿重启内核。"
assert isinstance(result, dict) and "sys_analyser" in result, (
    "当前 result 不是有效的 RQAlpha 结果。"
)

analyser = result["sys_analyser"]

portable_tables = {}
for table_name in [
    "portfolio",
    "benchmark_portfolio",
    "stock_account",
    "trades",
    "stock_positions",
]:
    table = analyser[table_name].copy()

    # trades 的索引名 datetime 与原字段 datetime 重名时，不重复插入索引。
    if table.index.name is not None and table.index.name in table.columns:
        table = table.reset_index(drop=True)
    else:
        table = table.reset_index()

    portable_tables[table_name] = json.loads(
        table.to_json(orient="split", date_format="iso")
    )

portable_result = {
    "format": "rqalpha_analyser_portable_v1",
    "portfolio_name": "Q5_factor_high",
    "summary": json.loads(
        json.dumps(analyser["summary"], default=str)
    ),
}
portable_result.update(portable_tables)

with open(Q5_PORTABLE_RESULT, "wb") as file:
    pickle.dump(portable_result, file, protocol=4)

print("Q5 强制清仓可移植结果已保存：", Q5_PORTABLE_RESULT)
print("文件大小（MB）：", round(Q5_PORTABLE_RESULT.stat().st_size / 1024**2, 2))

In [ ]:
# 09.0：Python 3.9 研究分析环境初始化 ----
from pathlib import Path
import json
import pickle

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)

assert PROJECT_ROOT.exists(), f"项目根目录不存在：{PROJECT_ROOT}"

TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"
BACKTEST_DIR = PROJECT_ROOT / "data" / "backtests"

for directory in [TABLE_DIR, FIGURE_DIR, METADATA_DIR, BACKTEST_DIR]:
    assert directory.exists(), f"缺少目录：{directory}"

print("===== 09 Python 3.9 分析环境已启动 =====")
print("项目根目录：", PROJECT_ROOT)
print("数据回测目录：", BACKTEST_DIR)
print("说明：本阶段只读取可移植 pkl 与安全 CSV，不运行 RQAlpha。")

In [ ]:
# 09.3.3：强制清仓尾部敏感性结果核对与正式导出 ----
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"
BACKTEST_DIR = PROJECT_ROOT / "data" / "backtests"

FORCED_RESULT_FILES = {
    "Q5_factor_high": (
        BACKTEST_DIR
        / "09_rqalpha_q5_forced_exit_20d_portable_result.pkl"
    ),
    "Q1_factor_low": (
        BACKTEST_DIR
        / "09_rqalpha_q1_forced_exit_20d_portable_result.pkl"
    ),
    "Equal_weight_eligible_universe": (
        BACKTEST_DIR
        / "09_rqalpha_equal_weight_forced_exit_20d_portable_result.pkl"
    ),
}

ORIGINAL_PERFORMANCE_PATH = (
    TABLE_DIR / "08_rqalpha_oos_three_portfolio_performance.csv"
)

for path in list(FORCED_RESULT_FILES.values()) + [ORIGINAL_PERFORMANCE_PATH]:
    assert path.exists(), f"未找到所需文件：{path}"

forced_results = {}

for portfolio_name, result_file in FORCED_RESULT_FILES.items():
    with open(result_file, "rb") as file:
        payload = pickle.load(file)

    assert payload.get("format") == "rqalpha_analyser_portable_v1", (
        f"{portfolio_name} 的可移植结果格式不正确。"
    )

    portfolio_table = pd.DataFrame(
        payload["portfolio"]["data"],
        columns=payload["portfolio"]["columns"],
    )
    benchmark_table = pd.DataFrame(
        payload["benchmark_portfolio"]["data"],
        columns=payload["benchmark_portfolio"]["columns"],
    )
    trades_table = pd.DataFrame(
        payload["trades"]["data"],
        columns=payload["trades"]["columns"],
    )

    portfolio_date_column = next(
        column
        for column in ["date", "datetime", "trading_datetime", "index"]
        if column in portfolio_table.columns
    )
    benchmark_date_column = next(
        column
        for column in ["date", "datetime", "trading_datetime", "index"]
        if column in benchmark_table.columns
    )
    trades_date_column = next(
        column
        for column in ["trading_datetime", "datetime", "date", "index"]
        if column in trades_table.columns
    )

    portfolio_table[portfolio_date_column] = (
    pd.to_datetime(
        portfolio_table[portfolio_date_column],
        utc=True,
    )
    .dt.tz_localize(None)
    .dt.normalize()
)

    benchmark_table[benchmark_date_column] = (
    pd.to_datetime(
        benchmark_table[benchmark_date_column],
        utc=True,
    )
        .dt.tz_localize(None)
        .dt.normalize()
)

    trades_table[trades_date_column] = (
    pd.to_datetime(
        trades_table[trades_date_column],
        utc=True,
    )
        .dt.tz_localize(None)
        .dt.normalize()
)
    portfolio_table = (
        portfolio_table
        .set_index(portfolio_date_column)
        .sort_index()
    )
    benchmark_table = (
        benchmark_table
        .set_index(benchmark_date_column)
        .sort_index()
    )
    trades_table = (
        trades_table
        .set_index(trades_date_column)
        .sort_index()
    )

    forced_results[portfolio_name] = {
        "portfolio": portfolio_table,
        "benchmark": benchmark_table,
        "trades": trades_table,
        "summary": payload["summary"],
    }

original_performance = pd.read_csv(ORIGINAL_PERFORMANCE_PATH)
original_performance = original_performance.set_index("portfolio")

ORIGINAL_END_DATE = pd.Timestamp("2026-02-02")
sensitivity_records = []

for portfolio_name, result_bundle in forced_results.items():
    portfolio_nav = result_bundle["portfolio"]
    benchmark_nav = result_bundle["benchmark"]
    trades = result_bundle["trades"]

    strategy_nav = portfolio_nav["unit_net_value"].astype(float)
    benchmark_series = benchmark_nav["unit_net_value"].astype(float)

    cutoff_strategy_nav = strategy_nav.loc[:ORIGINAL_END_DATE].iloc[-1]
    cutoff_benchmark_nav = benchmark_series.loc[:ORIGINAL_END_DATE].iloc[-1]

    final_strategy_nav = strategy_nav.iloc[-1]
    final_benchmark_nav = benchmark_series.iloc[-1]

    post_exit_trades = trades.loc[
        (trades.index > ORIGINAL_END_DATE)
        & (trades.index <= strategy_nav.index.max())
    ].copy()

    if len(post_exit_trades) > 0:
        post_exit_notional = float(
            (
                post_exit_trades["last_quantity"].abs()
                * post_exit_trades["last_price"].abs()
            ).sum()
        )
        post_exit_cost = float(
            post_exit_trades["commission"].sum()
            + post_exit_trades["tax"].sum()
        )
    else:
        post_exit_notional = 0.0
        post_exit_cost = 0.0

    final_market_value = float(portfolio_nav["market_value"].iloc[-1])
    final_cash = float(portfolio_nav["cash"].iloc[-1])

    original_total_return = float(
        original_performance.loc[portfolio_name, "total_return"]
    )

    sensitivity_records.append(
        {
            "portfolio": portfolio_name,
            "original_backtest_end_date": ORIGINAL_END_DATE.strftime("%Y-%m-%d"),
            "forced_exit_backtest_end_date": (
                strategy_nav.index.max().strftime("%Y-%m-%d")
            ),
            "original_total_return": original_total_return,
            "total_return_at_original_end_date": (
                float(cutoff_strategy_nav - 1.0)
            ),
            "forced_exit_final_total_return": (
                float(final_strategy_nav - 1.0)
            ),
            "post_exit_tail_return": float(
                final_strategy_nav / cutoff_strategy_nav - 1.0
            ),
            "forced_exit_final_benchmark_return": float(
                final_benchmark_nav - 1.0
            ),
            "post_exit_tail_benchmark_return": float(
                final_benchmark_nav / cutoff_benchmark_nav - 1.0
            ),
            "final_market_value": final_market_value,
            "final_cash": final_cash,
            "fully_liquidated_at_forced_exit_end": bool(
                abs(final_market_value) < 1.0
            ),
            "post_exit_trade_record_count": int(len(post_exit_trades)),
            "post_exit_filled_notional": post_exit_notional,
            "post_exit_transaction_cost": post_exit_cost,
        }
    )

forced_exit_sensitivity = pd.DataFrame(sensitivity_records)

# Q5 相对 Q1 / 等权券池：统一按首个可见净值日标准化，避免首日成交差异干扰。
relative_records = []

q5_nav = forced_results["Q5_factor_high"]["portfolio"]["unit_net_value"].astype(float)

for comparison_name in [
    "Q1_factor_low",
    "Equal_weight_eligible_universe",
]:
    comparison_nav = (
        forced_results[comparison_name]["portfolio"]["unit_net_value"]
        .astype(float)
    )

    common_dates = q5_nav.index.intersection(comparison_nav.index)
    q5_normalized = q5_nav.loc[common_dates] / q5_nav.loc[common_dates].iloc[0]
    comparison_normalized = (
        comparison_nav.loc[common_dates]
        / comparison_nav.loc[common_dates].iloc[0]
    )

    relative_nav = q5_normalized / comparison_normalized
    relative_at_original_end = relative_nav.loc[:ORIGINAL_END_DATE].iloc[-1]
    relative_at_forced_exit_end = relative_nav.iloc[-1]

    relative_records.append(
        {
            "comparison": f"Q5_factor_high_vs_{comparison_name}",
            "relative_return_at_original_end_date": float(
                relative_at_original_end - 1.0
            ),
            "relative_return_at_forced_exit_end": float(
                relative_at_forced_exit_end - 1.0
            ),
            "relative_return_change_during_exit_tail": float(
                relative_at_forced_exit_end / relative_at_original_end - 1.0
            ),
        }
    )

relative_sensitivity = pd.DataFrame(relative_records)

# 安全研究产物
forced_exit_sensitivity.to_csv(
    TABLE_DIR / "09_rqalpha_forced_exit_sensitivity_performance.csv",
    index=False,
)
relative_sensitivity.to_csv(
    TABLE_DIR / "09_rqalpha_forced_exit_relative_comparison.csv",
    index=False,
)

rules = {
    "analysis_type": "forced_liquidation_tail_sensitivity",
    "original_backtest_end_date": "2026-02-02",
    "forced_exit_target_dates": "2026-02-03_to_2026-03-10",
    "forced_exit_rule": "所有组合在追加的 20 个交易日每日目标权重为零。",
    "comparison_rule": (
        "三个组合使用相同的原始权重、成本、撮合、涨跌停与成交量限制；"
        "仅改变期末后的强制退出时间窗。"
    ),
    "interpretation_rule": (
        "强制退出后若最终 market_value 为零，说明在窗口内完成退出；"
        "若不为零，则仍应视为未完全可退出。"
    ),
}
with open(
    METADATA_DIR / "09_rqalpha_forced_exit_sensitivity_rules.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(rules, file, ensure_ascii=False, indent=2)

# 图表
plt.style.use("default")
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

for portfolio_name, result_bundle in forced_results.items():
    nav_series = result_bundle["portfolio"]["unit_net_value"].astype(float)
    nav_series = nav_series / nav_series.iloc[0]

    axes[0].plot(nav_series.index, nav_series, label=portfolio_name)

axes[0].axvline(
    ORIGINAL_END_DATE,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Original end",
)
axes[0].set_title("Forced-Exit Sensitivity: Normalized NAV")
axes[0].set_ylabel("Normalized NAV")
axes[0].legend(fontsize=8)

axes[1].bar(
    forced_exit_sensitivity["portfolio"],
    forced_exit_sensitivity["post_exit_tail_return"] * 100,
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_title("Return During Forced-Exit Tail")
axes[1].set_ylabel("Return (%)")

axes[2].bar(
    forced_exit_sensitivity["portfolio"],
    forced_exit_sensitivity["final_market_value"] / 1_000_000,
)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].tick_params(axis="x", rotation=25)
axes[2].set_title("Residual Market Value at Final Date")
axes[2].set_ylabel("Million CNY")

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "09_rqalpha_forced_exit_sensitivity.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("===== 强制清仓敏感性：策略表现 =====")
display(forced_exit_sensitivity)

print("\n===== 强制清仓敏感性：Q5 相对表现 =====")
display(relative_sensitivity)

print("\n===== 已保存 =====")
print(TABLE_DIR / "09_rqalpha_forced_exit_sensitivity_performance.csv")
print(TABLE_DIR / "09_rqalpha_forced_exit_relative_comparison.csv")
print(METADATA_DIR / "09_rqalpha_forced_exit_sensitivity_rules.json")
print(FIGURE_DIR / "09_rqalpha_forced_exit_sensitivity.png")

In [ ]:
# 09.4：冻结最终研究决策并更新研究协议 ----
HAC_PATH = (
    TABLE_DIR
    / "09_integrated_profitability_hac_stability_summary.csv"
)
FORCED_PERFORMANCE_PATH = (
    TABLE_DIR
    / "09_rqalpha_forced_exit_sensitivity_performance.csv"
)
FORCED_RELATIVE_PATH = (
    TABLE_DIR
    / "09_rqalpha_forced_exit_relative_comparison.csv"
)
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "09_RESEARCH_PROTOCOL.md"

for path in [
    HAC_PATH,
    FORCED_PERFORMANCE_PATH,
    FORCED_RELATIVE_PATH,
    PROTOCOL_PATH,
]:
    assert path.exists(), f"缺少收尾所需文件：{path}"

hac_summary = pd.read_csv(HAC_PATH)
forced_performance = pd.read_csv(FORCED_PERFORMANCE_PATH)
forced_relative = pd.read_csv(FORCED_RELATIVE_PATH)

oos_hac = hac_summary.loc[
    hac_summary["period"] == "out_of_sample_2023_2025"
].iloc[0]

q5_vs_q1 = forced_relative.loc[
    forced_relative["comparison"]
    == "Q5_factor_high_vs_Q1_factor_low"
].iloc[0]

q5_vs_equal_weight = forced_relative.loc[
    forced_relative["comparison"]
    == "Q5_factor_high_vs_Equal_weight_eligible_universe"
].iloc[0]

all_fully_liquidated = (
    forced_performance["fully_liquidated_at_forced_exit_end"]
    .astype(str)
    .str.lower()
    .eq("true")
    .all()
)

confirmation_conditions = {
    "oos_rank_ic_hac_significant": (
        float(oos_hac["rank_ic_hac_p_value"]) < 0.05
    ),
    "oos_high_minus_low_hac_significant": (
        float(oos_hac["high_minus_low_return_hac_p_value"]) < 0.05
    ),
    "q5_outperforms_equal_weight_after_forced_exit": (
        float(
            q5_vs_equal_weight[
                "relative_return_at_forced_exit_end"
            ]
        ) > 0
    ),
    "all_portfolios_fully_liquidated": bool(all_fully_liquidated),
}

if all(confirmation_conditions.values()):
    final_decision = "confirmed_incremental_selection_alpha"
else:
    final_decision = "candidate_not_confirmed"

scorecard = pd.DataFrame(
    [
        {
            "metric": "oos_rank_ic_hac_p_value",
            "value": float(oos_hac["rank_ic_hac_p_value"]),
        },
        {
            "metric": "oos_q5_minus_q1_hac_p_value",
            "value": float(
                oos_hac["high_minus_low_return_hac_p_value"]
            ),
        },
        {
            "metric": "q5_vs_q1_relative_return_after_forced_exit",
            "value": float(
                q5_vs_q1["relative_return_at_forced_exit_end"]
            ),
        },
        {
            "metric": "q5_vs_equal_weight_relative_return_after_forced_exit",
            "value": float(
                q5_vs_equal_weight[
                    "relative_return_at_forced_exit_end"
                ]
            ),
        },
        {
            "metric": "all_portfolios_fully_liquidated",
            "value": bool(all_fully_liquidated),
        },
        {
            "metric": "final_research_decision",
            "value": final_decision,
        },
    ]
)

decision_log = pd.DataFrame(
    [
        {
            "evidence": "严格样本外 RankIC HAC 检验",
            "finding": (
                "RankIC HAC p={:.4f}，未达到显著性门槛。".format(
                    float(oos_hac["rank_ic_hac_p_value"])
                )
            ),
            "decision_implication": "不确认稳定横截面排序 Alpha。",
        },
        {
            "evidence": "严格样本外 Q5-Q1 HAC 检验",
            "finding": (
                "Q5-Q1 HAC p={:.4f}，未达到显著性门槛。".format(
                    float(
                        oos_hac[
                            "high_minus_low_return_hac_p_value"
                        ]
                    )
                )
            ),
            "decision_implication": "不确认显著多空收益。",
        },
        {
            "evidence": "RQAlpha 强制清仓后 Q5 相对 Q1",
            "finding": (
                "相对收益为 {:.4%}。".format(
                    float(
                        q5_vs_q1[
                            "relative_return_at_forced_exit_end"
                        ]
                    )
                )
            ),
            "decision_implication": "保留高低分组的正向排序证据。",
        },
        {
            "evidence": "RQAlpha 强制清仓后 Q5 相对等权券池",
            "finding": (
                "相对收益为 {:.4%}，仍未转正。".format(
                    float(
                        q5_vs_equal_weight[
                            "relative_return_at_forced_exit_end"
                        ]
                    )
                )
            ),
            "decision_implication": "不确认增量选股 Alpha。",
        },
        {
            "evidence": "期末强制退出可执行性",
            "finding": (
                "三个组合均在 20 个追加交易日内完成清仓。"
            ),
            "decision_implication": (
                "原结论不依赖 2026-02-02 跌停导致的市值计价。"
            ),
        },
        {
            "evidence": "最终研究状态",
            "finding": final_decision,
            "decision_implication": (
                "冻结因子定义、权重、样本划分与策略规则；"
                "不得依据既有样本外结果继续优化。"
            ),
        },
    ]
)

scorecard.to_csv(
    TABLE_DIR / "09_final_research_scorecard.csv",
    index=False,
)
decision_log.to_csv(
    TABLE_DIR / "09_final_research_decision_log.csv",
    index=False,
)

final_rules = {
    "final_research_decision": final_decision,
    "confirmation_conditions": confirmation_conditions,
    "decision_rule": (
        "只有 RankIC 与 Q5-Q1 的样本外 HAC 检验均显著、"
        "Q5 强制清仓后超过等权可投资券池、"
        "且全部组合完成清仓，才确认增量选股 Alpha。"
    ),
    "freeze_rule": (
        "不使用 2023-2025 样本外结果调整组件权重、"
        "覆盖率门槛、持有期、股票池或交易规则。"
    ),
}

with open(
    METADATA_DIR / "09_final_research_decision_rules.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(final_rules, file, ensure_ascii=False, indent=2)

protocol_marker = "## 最终执行稳健性更新"
protocol_update = f"""
## 最终执行稳健性更新

对 Q5、Q1 与等权可投资券池统一追加 20 个交易日的每日零权重目标，以检验 2026-02-02 跌停拒单后的实际退出影响。

三个组合均完成清仓，期末残余市值均为零。Q5 的退出尾部收益为 {float(forced_performance.loc[forced_performance["portfolio"] == "Q5_factor_high", "post_exit_tail_return"].iloc[0]):.4%}；Q1 为 {float(forced_performance.loc[forced_performance["portfolio"] == "Q1_factor_low", "post_exit_tail_return"].iloc[0]):.4%}；等权可投资券池为 {float(forced_performance.loc[forced_performance["portfolio"] == "Equal_weight_eligible_universe", "post_exit_tail_return"].iloc[0]):.4%}。

强制清仓后，Q5 相对 Q1 的收益为 {float(q5_vs_q1["relative_return_at_forced_exit_end"]):.4%}，相对等权可投资券池为 {float(q5_vs_equal_weight["relative_return_at_forced_exit_end"]):.4%}。因此，期末未成交并不改变最终结论：当前等权合成对照版本具有高低分组排序证据，但未确认存在超过等权可投资券池的独立增量选股 Alpha。

最终研究状态：`{final_decision}`。
""".strip()

protocol_content = PROTOCOL_PATH.read_text(encoding="utf-8")

if protocol_marker not in protocol_content:
    PROTOCOL_PATH.write_text(
        protocol_content.rstrip() + "\n\n" + protocol_update + "\n",
        encoding="utf-8",
    )
    protocol_status = "已追加强制清仓稳健性结论。"
else:
    protocol_status = "协议中已有该更新，未重复写入。"

print("===== 最终研究决策 =====")
print("状态：", final_decision)
print("协议更新：", protocol_status)

display(scorecard)
display(decision_log)

print("\n已保存：")
print(TABLE_DIR / "09_final_research_scorecard.csv")
print(TABLE_DIR / "09_final_research_decision_log.csv")
print(METADATA_DIR / "09_final_research_decision_rules.json")
print(PROTOCOL_PATH)